<a href="https://colab.research.google.com/github/italobotelho/PI2026.1/blob/main/02_ETL_Geolocalizacao_CNES.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧬 **Projeto SIEST: Sistema de Inteligência Epidemiológica e Socioespacial**
## Notebook 02: Geolocalização e Cruzamento Espacial (Camada Silver)

Este notebook é responsável por elevar os dados da Camada *Bronze* (dados brutos de morbidade) para a Camada *Silver* (dados refinados e georreferenciados).

**Estratégia Arquitetural e Resolução de Problemas:**
Durante a extração via FTP governamental, os servidores do DataSUS apresentaram bloqueios por limite de requisições (*Rate Limiting*) e ausência das coordenadas espaciais na base transacional de estabelecimentos.

Para contornar esta instabilidade e garantir a resiliência do *pipeline*, a equipe adotou a criação de um **Contrato de Dados Rígido (*Strict Schema*)** atrelado a uma fonte estática: o arquivo CSV do Cadastro Nacional de Estabelecimentos de Saúde (CNES), obtido via Portal de Dados Abertos. Esta abordagem permite o cruzamento determinístico das coordenadas geográficas (Latitude/Longitude) com os casos notificados.

### 1. Importação de Bibliotecas e Contrato de Dados

In [1]:
import pandas as pd
import os

# Mapeamento estrito das colunas esperadas no CSV oficial do governo
COL_MUNICIPIO = 'CO_IBGE'
COL_CNES = 'CO_CNES'
COL_NOME = 'NO_FANTASIA'
COL_LAT = 'NU_LATITUDE'
COL_LON = 'NU_LONGITUDE'

# Definição dos caminhos das camadas do Data Lake
PASTA_BRONZE = 'dados_sinan_parquet'
PASTA_SILVER = 'dados_processados_cnes_sinan_parquet'
CAMINHO_CNES_CSV = 'cnes_estabelecimentos.csv'

os.makedirs(PASTA_SILVER, exist_ok=True)

### 2. Leitura e Preparação das Bases (Bronze e Estática)

Nesta etapa, o *script* consolida todos os ficheiros particionados da Camada Bronze num único *DataFrame* em memória e carrega a base territorial do CNES, aplicando filtros de formatação para evitar a perda de zeros à esquerda nos códigos identificadores.

In [2]:
print("A carregar as partições do SINAN (Camada Bronze)...")

# Verifica se a camada Bronze existe e contém dados
if not os.path.exists(PASTA_BRONZE):
    raise FileNotFoundError(f"A pasta '{PASTA_BRONZE}' não foi encontrada. Rode o Notebook 01 primeiro.")

ficheiros_sinan = [os.path.join(PASTA_BRONZE, f) for f in os.listdir(PASTA_BRONZE) if f.endswith('.parquet')]
df_sinan = pd.concat([pd.read_parquet(f) for f in ficheiros_sinan], ignore_index=True)
print(f"Total de {len(df_sinan)} notificações epidemiológicas prontas para cruzamento.")

print("\nA ler a base territorial estática do CNES (Ficheiro CSV)...")
# Leitura forçada como 'string' para preservar a integridade dos códigos
df_cnes = pd.read_csv(CAMINHO_CNES_CSV, sep=';', encoding='latin1', dtype=str)

# Filtra apenas as unidades de Campinas (Prefixo 350950)
df_cnes_campinas = df_cnes[df_cnes[COL_MUNICIPIO].str.startswith('350950', na=False)].copy()
df_cnes_campinas = df_cnes_campinas[[COL_CNES, COL_NOME, COL_LAT, COL_LON]]

# Padroniza a nomenclatura para o futuro banco NoSQL
df_cnes_campinas.rename(columns={
    COL_CNES: 'ID_UNIDADE',
    COL_LAT: 'LATITUDE',
    COL_LON: 'LONGITUDE'
}, inplace=True)

print(f"Total de {len(df_cnes_campinas)} estabelecimentos de saúde mapeados em Campinas.")

A carregar as partições do SINAN (Camada Bronze)...
Total de 410846 notificações epidemiológicas prontas para cruzamento.

A ler a base territorial estática do CNES (Ficheiro CSV)...
Total de 4867 estabelecimentos de saúde mapeados em Campinas.


### 2. Cruzamento Espacial (Left Join) e Persistência

A chave primária de ligação entre a base de morbidade e a base territorial é o Código da Unidade (`ID_UNIDADE`). O código aplica rotinas de limpeza (Regex) para higienizar potenciais casas decimais espúrias antes de executar a junção relacional à esquerda (*Left Join*).

O resultado final é persistido na Camada Silver.

In [8]:
print("A executar a higienização de chaves e o cruzamento espacial...")

# Limpeza rigorosa das chaves de ligação (remoção de .0 perdidos por tipagem flutuante)
df_sinan['ID_UNIDADE'] = df_sinan['ID_UNIDADE'].astype(str).str.replace(r'\.0$', '', regex=True).replace('nan', pd.NA)
df_cnes_campinas['ID_UNIDADE'] = df_cnes_campinas['ID_UNIDADE'].astype(str).str.replace(r'\.0$', '', regex=True)

# Execução do Left Join: Preserva todos os casos, agregando a Lat/Lon quando a unidade for encontrada
df_siest_geo = pd.merge(df_sinan, df_cnes_campinas, on='ID_UNIDADE', how='left')

# Salva o arquivo Mestre na Camada Silver
ficheiro_final = f'{PASTA_SILVER}/siest_casos_geolocalizados.parquet'
df_siest_geo.to_parquet(ficheiro_final, index=False, compression='snappy')

print(f"\n✅ PIPELINE SILVER FECHADO COM SUCESSO!")
print(f"Base Mestre Georreferenciada guardada em: '{ficheiro_final}'")

A executar a higienização de chaves e o cruzamento espacial...

✅ PIPELINE SILVER FECHADO COM SUCESSO!
Base Mestre Georreferenciada guardada em: 'dados_processados_cnes_sinan_parquet/siest_casos_geolocalizados.parquet'
